In [ ]:
# Feature 1: Chat Parser
# Reads hostel_bois.txt and extracts all real messages

import numpy as np
from datetime import datetime

# Counters for skipped lines
system_count = 0
media_count = 0
deleted_count = 0

# List to store parsed messages
messages = []  # Each item: {'timestamp': str, 'sender': str, 'text': str, 'datetime': datetime}

# Read the file
with open('/content/hostel_bois.txt', 'r', encoding='utf-8') as f:
    lines = f.readlines()

#  Parse line by line
for line in lines:
    line = line.strip()

    # Skip empty lines
    if not line:
        continue

    # Check if line starts with a date pattern (DD/MM/YY)
    # First 8 chars should look like: 12/04/24
    if not (len(line) > 8 and line[2] == '/' and line[5] == '/'):
        # Multi-line message continuation — append to last message
        if messages:
            messages[-1]['text'] += ' ' + line
        continue

    # Split into timestamp and the rest
    # Format: 12/04/24, 23:14 - Rahul: bhai aaj kya scene
    try:
        parts = line.split(' - ', 1)
        if len(parts) < 2:
            system_count += 1
            continue

        timestamp_str = parts[0]   # "12/04/24, 23:14"
        rest = parts[1]            # "Rahul: bhai aaj kya scene"

        # Check if it's a system message (no colon for sender)
        if ': ' not in rest:
            system_count += 1
            continue

        sender, text = rest.split(': ', 1)

        # Handle special message types
        if '<Media omitted>' in text:
            media_count += 1
            # Still count as a message from sender, but flag it
            messages.append({
                'timestamp': timestamp_str,
                'sender': sender.strip(),
                'text': text.strip(),
                'datetime': datetime.strptime(timestamp_str.strip(), '%d/%m/%y, %H:%M'),
                'is_media': True,
                'is_deleted': False
            })
            continue

        if 'This message was deleted' in text:
            deleted_count += 1
            messages.append({
                'timestamp': timestamp_str,
                'sender': sender.strip(),
                'text': text.strip(),
                'datetime': datetime.strptime(timestamp_str.strip(), '%d/%m/%y, %H:%M'),
                'is_media': False,
                'is_deleted': True
            })
            continue

        # Normal message
        messages.append({
            'timestamp': timestamp_str,
            'sender': sender.strip(),
            'text': text.strip(),
            'datetime': datetime.strptime(timestamp_str.strip(), '%d/%m/%y, %H:%M'),
            'is_media': False,
            'is_deleted': False
        })

    except Exception as e:
        system_count += 1
        continue

# Checkpoint print
participants = list(set(m['sender'] for m in messages))
real_messages = [m for m in messages if not m['is_media'] and not m['is_deleted']]

print(f" Successfully parsed {len(messages)} total messages")
print(f" Real messages (no media/deleted): {len(real_messages)}")
print(f" Participants ({len(participants)}): {', '.join(sorted(participants))}")
print(f"  Skipped — System: {system_count} | Media: {media_count} | Deleted: {deleted_count}")
print()
print(" First 3 messages")
for m in messages[:3]:
    print(f"  [{m['timestamp']}] {m['sender']}: {m['text']}")
print()
print(" Last 3 messages ")
for m in messages[-3:]:
    print(f"  [{m['timestamp']}] {m['sender']}: {m['text']}")

 Successfully parsed 3174 total messages
 Real messages (no media/deleted): 3127
 Participants (6): Aman, Karan, Neha, Priya, Rahul, Vikas
  Skipped — System: 4 | Media: 32 | Deleted: 15

 First 3 messages
  [01/04/24, 01:17] Rahul: scene fix
  [01/04/24, 01:17] Rahul: haan
  [01/04/24, 01:18] Rahul: kya scene

 Last 3 messages 
  [30/05/24, 21:17] Aman: the existential dread is back
  [30/05/24, 21:30] Karan: Long day guys, woke up at six for that placement workshop which started at eight by the way classic, then ran around campus collecting signatures for the project document, ate lunch at four PM standing in the corridor, came back to hostel and realized I forgot my charger in the lab.
  [30/05/24, 23:31] Aman: anyone awake?


In [ ]:
# Feature 2: Group Overview
# Computes headline stats and per-person message counts

# Date range
first_date = messages[0]['datetime']
last_date = messages[-1]['datetime']
total_days = (last_date - first_date).days + 1

#  Per-person message count (all messages including media/deleted)
msg_count = {}
for m in messages:
    sender = m['sender']
    msg_count[sender] = msg_count.get(sender, 0) + 1

total_msgs = len(messages)

# Sort by count descending
sorted_senders = sorted(msg_count.keys(), key=lambda x: msg_count[x], reverse=True)

# Bar renderer helper (reuse this throughout the project)
def make_bar(value, max_value, bar_width=20):
    filled = int((value / max_value) * bar_width)
    return '█' * filled + '░' * (bar_width - filled)

max_count = msg_count[sorted_senders[0]]

#  Print the report section
print('=' * 62)
print('  GROUPDNA REPORT — "Hostel Bois 4ever"')
print(f'  {total_days} days  •  {total_msgs:,} messages  •  {len(participants)} members')
print('=' * 62)
print(f'  {"Period":<20}: {first_date.strftime("%d %B %Y")} to {last_date.strftime("%d %B %Y")}')
print()
print(f'  {"MESSAGES PER PERSON":}')
print(f'  {"-" * 55}')
for sender in sorted_senders:
    count = msg_count[sender]
    pct = (count / total_msgs) * 100
    bar = make_bar(count, max_count, bar_width=20)
    print(f'  {sender:<8} {bar}  {count:>4}  ({pct:>4.1f}%)')

  GROUPDNA REPORT — "Hostel Bois 4ever"
  60 days  •  3,174 messages  •  6 members
  Period              : 01 April 2024 to 30 May 2024

  MESSAGES PER PERSON
  -------------------------------------------------------
  Rahul    ████████████████████   953  (30.0%)
  Priya    ███████████████░░░░░   718  (22.6%)
  Neha     █████████████░░░░░░░   635  (20.0%)
  Aman     ██████████░░░░░░░░░░   490  (15.4%)
  Karan    ███████░░░░░░░░░░░░░   354  (11.2%)
  Vikas    ░░░░░░░░░░░░░░░░░░░░    24  ( 0.8%)


In [ ]:
# Feature 3: Most Active Day & Hour + Top Words

# ── Stop words to exclude ──────────────────────────────────
STOP_WORDS = {
    # English basics
    'i', 'is', 'the', 'a', 'and', 'or', 'to', 'of', 'in', 'on',
    'for', 'it', 'this', 'that', 'was', 'are', 'be', 'at', 'by',
    'we', 'you', 'he', 'she', 'they', 'my', 'your', 'his', 'her',
    'an', 'so', 'if', 'but', 'not', 'do', 'did', 'have', 'has',
    'me', 'him', 'us', 'its', 'im', 'with', 'from', 'will', 'had',
    'just', 'no', 'yes', 'ok', 'okay', 'hi', 'hey', 'oh', 'ah',
    'also', 'then', 'than', 'when', 'what', 'how', 'who', 'why',
    'all', 'can', 'get', 'got', 'like', 'too', 'up', 'out', 'more',
    'about', 'been', 'some', 'there', 'their', 'them', 'these',
    'would', 'could', 'should', 'dont', 'didnt', 'isnt', 'wasnt',
    'media', 'omitted', 'message', 'deleted',
    # Common short fillers
    'one', 'two', 'am', 'pm', 'go', 'going', 'come', 'came',
    'see', 'say', 'said', 'know', 'think', 'want', 'need',
    'good', 'now', 'here', 'where', 'which', 'while', 'even',
    'day', 'time', 'back', 'way', 'made', 'make', 'very',
    'much', 'after', 'before', 'any', 'every', 'only', 'other',
    'our', 'into', 'over', 'such', 'new', 'first', 'last',
    'long', 'down', 'never', 'between', 'each', 'both', 'still',
    # Hindi filler words (romanized)
    'hai', 'hain', 'tha', 'thi', 'the', 'ko', 'ke', 'ka', 'ki',
    'se', 'pe', 'par', 'aur', 'ya', 'na', 'nahi', 'mat', 'ho',
    'hoga', 'kar', 'karo', 'raha', 'rahi', 'wala', 'wali',
    'mein', 'toh', 'bhi', 'hi', 'hu', 'hoo', 'ek',
}

PUNCTUATION = '.,!?;:\'"()[]{}…-–—/\\@#$%^&*+=<>~`|'

def clean_word(word):
    """Strip punctuation and return lowercase word."""
    return word.lower().strip(PUNCTUATION)

#  Most active DAY
day_count = {}
for m in messages:
    day = m['datetime'].strftime('%d/%m/%Y')
    day_count[day] = day_count.get(day, 0) + 1

busiest_day_key = max(day_count, key=lambda x: day_count[x])
busiest_day_count = day_count[busiest_day_key]
# Convert to readable format
busiest_day_dt = datetime.strptime(busiest_day_key, '%d/%m/%Y')
busiest_day_str = busiest_day_dt.strftime('%d %B %Y')

# Most active HOUR
hour_count = {}
for m in messages:
    hour = m['datetime'].hour
    hour_count[hour] = hour_count.get(hour, 0) + 1

busiest_hour = max(hour_count, key=lambda x: hour_count[x])
avg_per_day = hour_count[busiest_hour] / total_days

#  Word frequency (real messages only, no media/deleted)
word_freq = {}          # group-wide
word_freq_per_person = {s: {} for s in participants}

for m in messages:
    if m['is_media'] or m['is_deleted']:
        continue
    words = m['text'].split()
    for raw in words:
        w = clean_word(raw)
        if len(w) < 2 or w in STOP_WORDS or not w.isalpha():
            continue
        word_freq[w] = word_freq.get(w, 0) + 1
        sender = m['sender']
        word_freq_per_person[sender][w] = word_freq_per_person[sender].get(w, 0) + 1

# Top 10 group words
top_words = sorted(word_freq.keys(), key=lambda x: word_freq[x], reverse=True)[:10]
max_word_count = word_freq[top_words[0]]

# Top 5 words per person
top_per_person = {}
for sender in participants:
    wf = word_freq_per_person[sender]
    top_per_person[sender] = sorted(wf.keys(), key=lambda x: wf[x], reverse=True)[:5]

# Print
print(f'  {"Busiest day":<22}: {busiest_day_str} ({busiest_day_count} messages)')
print(f'  {"Busiest hour":<22}: {busiest_hour:02d}:00 - {busiest_hour+1:02d}:00  (avg {avg_per_day:.0f} msgs/day)')
print()
print("  THIS GROUP'S FAVOURITE WORDS")
print(f'  {"-" * 50}')
for word in top_words:
    count = word_freq[word]
    bar = make_bar(count, max_word_count, bar_width=22)
    print(f'  {word:<10} {bar}  {count}')

print()
print("  TOP 5 WORDS PER PERSON")
print(f'  {"-" * 50}')
for sender in sorted_senders:
    words_str = ', '.join(top_per_person[sender])
    print(f'  {sender:<8}: {words_str}')
# Debug: top 5 busiest days
print("\n  TOP 5 BUSIEST DAYS (debug)")
top_days = sorted(day_count.keys(), key=lambda x: day_count[x], reverse=True)[:5]
for d in top_days:
    dt = datetime.strptime(d, '%d/%m/%Y')
    print(f'  {dt.strftime("%d %B %Y")} : {day_count[d]} messages')

  Busiest day           : 04 May 2024 (76 messages)
  Busiest hour          : 18:00 - 19:00  (avg 4 msgs/day)

  THIS GROUP'S FAVOURITE WORDS
  --------------------------------------------------
  guys       ██████████████████████  318
  today      █████████████████░░░░░  257
  everyone   ████████████░░░░░░░░░░  187
  telling    ████████████░░░░░░░░░░  179
  bhai       ███████████░░░░░░░░░░░  160
  started    ██████████░░░░░░░░░░░░  150
  scene      ██████████░░░░░░░░░░░░  145
  entire     ██████████░░░░░░░░░░░░  145
  please     █████████░░░░░░░░░░░░░  141
  anyone     █████████░░░░░░░░░░░░░  139

  TOP 5 WORDS PER PERSON
  --------------------------------------------------
  Rahul   : bhai, scene, kya, yaar, ja
  Priya   : please, everyone, aman, anyone, take
  Neha    : guys, cant, today, wait, believe
  Aman    : sleep, anyone, wonder, night, does
  Karan   : telling, today, started, entire, guys
  Vikas   : haha, sorry, busy, haan, exam

  TOP 5 BUSIEST DAYS (debug)
  04 May 2024 

In [ ]:
# Feature 4: Activity Heatmap using NumPy
# Builds a 6x24 matrix: rows = participants, columns = hours (0-23)

#  Build the matrix
# We need a fixed order for participants (rows)
person_order = sorted_senders   # already sorted by message count: Rahul, Priya, Neha, Aman, Karan, Vikas
person_index = {name: i for i, name in enumerate(person_order)}

heatmap = np.zeros((len(person_order), 24), dtype=int)

for m in messages:
    row = person_index[m['sender']]
    col = m['datetime'].hour
    heatmap[row][col] += 1

# Render the heatmap
def render_heatmap(matrix, row_labels):
    """Renders a 2D numpy matrix as a text heatmap using block characters."""
    shade = ['. ', '░ ', '▒ ', '█ ']

    # Header: hour labels every 3 hours
    header = '  ' + ''.join(f'{h:02d}' + ' ' for h in range(0, 24, 3))
    print(f'  {"":10}  {header}')
    print(f'  {"-" * 65}')

    for i, label in enumerate(row_labels):
        row = matrix[i]
        row_max = row.max()
        if row_max == 0:
            row_max = 1  # avoid division by zero for silent members

        cells = ''
        for h in range(24):
            ratio = row[h] / row_max
            if ratio == 0:
                cells += shade[0]
            elif ratio <= 0.25:
                cells += shade[1]
            elif ratio <= 0.50:
                cells += shade[2]
            else:
                cells += shade[3]

            # Add spacing at every 3-hour mark for readability
            if (h + 1) % 3 == 0:
                cells += ' '

        print(f'  {label:<10}  {cells}')
# AI generated code
#  Some NumPy stats
print('  ACTIVITY HEATMAP (hour of day, columns 00 to 23)')
print()
render_heatmap(heatmap, person_order)

print()
print('  NUMPY QUICK STATS')
print(f'  {"-" * 50}')
for i, sender in enumerate(person_order):
    row = heatmap[i]
    peak_hour = int(row.argmax())
    total = int(row.sum())
    # Night owl check: hours 23,0,1,2,3,4
    night_cols = list(range(23, 24)) + list(range(0, 5))
    night_msgs = int(sum(heatmap[i][h] for h in night_cols))
    night_pct = (night_msgs / total * 100) if total > 0 else 0
    print(f'  {sender:<8}  peak hour: {peak_hour:02d}:00  |  night msgs: {night_msgs} ({night_pct:.1f}%)')

print()
print('  RAW NUMPY MATRIX (rows=people, cols=hours)')
print(f'  Shape: {heatmap.shape}')
print(f'  Total messages in matrix: {heatmap.sum()}')
print()
# Print the matrix neatly
hour_header = '        ' + ''.join(f'{h:>4}' for h in range(24))
print(f'  {hour_header}')
for i, sender in enumerate(person_order):
    row_str = ''.join(f'{v:>4}' for v in heatmap[i])
    print(f'  {sender:<8}{row_str}')

  ACTIVITY HEATMAP (hour of day, columns 00 to 23)

                00 03 06 09 12 15 18 21 
  -----------------------------------------------------------------
  Rahul       ░ ░ ░  ░ ░ ░  ░ ░ ░  ░ ░ ░  █ ▒ ▒  █ █ ▒  █ █ ▒  █ █ █  
  Priya       . . .  . . .  ░ ▒ █  █ █ █  █ █ █  ▒ ▒ █  █ █ █  ▒ ▒ ░  
  Neha        . . .  . . ▒  ░ ░ █  █ █ ▒  █ █ ▒  ░ █ █  █ █ █  ▒ ▒ ▒  
  Aman        █ █ █  █ █ .  . . .  . . .  . . ░  ░ ░ ░  ░ ░ ░  ░ . █  
  Karan       . . .  . . .  . ░ ▒  ▒ █ ▒  █ █ █  █ █ █  █ █ █  ▒ ░ ░  
  Vikas       . . .  . . .  . ▒ █  ▒ ▒ .  █ █ .  ▒ ▒ █  █ █ ▒  ▒ ▒ █  

  NUMPY QUICK STATS
  --------------------------------------------------
  Rahul     peak hour: 18:00  |  night msgs: 128 (13.4%)
  Priya     peak hour: 09:00  |  night msgs: 9 (1.3%)
  Neha      peak hour: 18:00  |  night msgs: 30 (4.7%)
  Aman      peak hour: 04:00  |  night msgs: 391 (79.8%)
  Karan     peak hour: 12:00  |  night msgs: 8 (2.3%)
  Vikas     peak hour: 08:00  |  night msgs: 2 (8.3%)

  RAW N

In [ ]:
# Feature 5: Response Times & Silent Streaks
# AI generated code
#  Average Response Time per person
# Logic: for each message, find the gap since the last message
# from a DIFFERENT sender. That gap = this person's response time.

response_gaps = {s: [] for s in participants}  # in seconds

for i in range(1, len(messages)):
    curr = messages[i]
    prev = messages[i - 1]

    # Only count if current sender is different from previous
    if curr['sender'] != prev['sender']:
        gap_seconds = (curr['datetime'] - prev['datetime']).total_seconds()

        # Ignore gaps over 24 hours (they went to sleep, not "responding")
        if 0 < gap_seconds < 24 * 3600:
            response_gaps[curr['sender']].append(gap_seconds)

# Compute average response time per person
avg_response = {}
for sender in participants:
    gaps = response_gaps[sender]
    if gaps:
        avg_response[sender] = sum(gaps) / len(gaps)
    else:
        avg_response[sender] = float('inf')

# Sort by fastest to slowest
sorted_by_response = sorted(avg_response.keys(), key=lambda x: avg_response[x])

def format_time(seconds):
    """Convert seconds to human-readable minutes or hours."""
    if seconds == float('inf'):
        return 'N/A'
    minutes = seconds / 60
    if minutes < 60:
        return f'{minutes:.1f} minutes'
    else:
        return f'{minutes/60:.1f} hours'

#  Silent Streaks per person
# Logic: for each person, find the longest run of consecutive
# days (within the chat period) where they sent 0 messages.

# Build set of active days per person
from datetime import timedelta

active_days = {s: set() for s in participants}
for m in messages:
    day_str = m['datetime'].strftime('%Y-%m-%d')
    active_days[m['sender']].add(day_str)

# Generate all days in the chat period
all_days = []
current = first_date.replace(hour=0, minute=0, second=0, microsecond=0)
end = last_date.replace(hour=0, minute=0, second=0, microsecond=0)
while current <= end:
    all_days.append(current.strftime('%Y-%m-%d'))
    current += timedelta(days=1)

# Find longest silent streak per person
silent_streaks = {}
streak_details = {}  # store start/end of longest streak

for sender in participants:
    active = active_days[sender]
    max_streak = 0
    current_streak = 0
    streak_start = None
    best_start = None
    best_end = None

    for day in all_days:
        if day not in active:
            if current_streak == 0:
                streak_start = day
            current_streak += 1
            if current_streak > max_streak:
                max_streak = current_streak
                best_start = streak_start
                best_end = day
        else:
            current_streak = 0
            streak_start = None

    silent_streaks[sender] = max_streak
    streak_details[sender] = (best_start, best_end)

# Sort by longest streak descending
sorted_by_streak = sorted(participants, key=lambda x: silent_streaks[x], reverse=True)

# Print
print('  RESPONSE PATTERNS')
print(f'  {"-" * 50}')
print(f' Fastest replier : {sorted_by_response[0]} (avg {format_time(avg_response[sorted_by_response[0]])})')
print(f' Slowest replier : {sorted_by_response[-1]} (avg {format_time(avg_response[sorted_by_response[-1]])})')
print()
print('  ALL RESPONSE TIMES (fastest to slowest)')
for sender in sorted_by_response:
    print(f'  {sender:<10} avg {format_time(avg_response[sender])}')

print()
print(' LONGEST SILENT STREAKS (consecutive days with zero messages)')
print(f'  {"-" * 50}')
for sender in sorted_by_streak:
    streak = silent_streaks[sender]
    days_active = len(active_days[sender])
    if streak == 0:
        print(f'  {sender:<10} : 0 days (never went silent)')
    else:
        s, e = streak_details[sender]
        s_fmt = datetime.strptime(s, '%Y-%m-%d').strftime('%d %b')
        e_fmt = datetime.strptime(e, '%Y-%m-%d').strftime('%d %b')
        print(f'  {sender:<10} : {streak} days  ({s_fmt} to {e_fmt})')

print()
print(' ACTIVE DAYS PER PERSON')
print(f'  {"-" * 50}')
for sender in sorted_senders:
    active = len(active_days[sender])
    silent = total_days - active
    silent_pct = (silent / total_days) * 100
    print(f'  {sender:<10} active {active:>2} / {total_days} days  |  silent {silent:>2} days ({silent_pct:.0f}%)')
# Debug: check how many days each person actually missed
print("\n  DEBUG - days with zero messages:")
for sender in participants:
    missed = [d for d in all_days if d not in active_days[sender]]
    print(f'  {sender:<10}: {len(missed)} silent days — {missed[:5]}')

  RESPONSE PATTERNS
  --------------------------------------------------
 Fastest replier : Rahul (avg 36.5 minutes)
 Slowest replier : Aman (avg 56.9 minutes)

  ALL RESPONSE TIMES (fastest to slowest)
  Rahul      avg 36.5 minutes
  Karan      avg 37.5 minutes
  Neha       avg 40.9 minutes
  Priya      avg 43.0 minutes
  Vikas      avg 46.3 minutes
  Aman       avg 56.9 minutes

 LONGEST SILENT STREAKS (consecutive days with zero messages)
  --------------------------------------------------
  Vikas      : 11 days  (23 Apr to 03 May)
  Priya      : 0 days (never went silent)
  Karan      : 0 days (never went silent)
  Rahul      : 0 days (never went silent)
  Neha       : 0 days (never went silent)
  Aman       : 0 days (never went silent)

 ACTIVE DAYS PER PERSON
  --------------------------------------------------
  Rahul      active 60 / 60 days  |  silent  0 days (0%)
  Priya      active 60 / 60 days  |  silent  0 days (0%)
  Neha       active 60 / 60 days  |  silent  0 days (0%)

In [ ]:
# Feature 6 & 7: Personality Archetype Detection
# One scoring function per archetype, assign highest scorer exclusively

real_msgs = [m for m in messages if not m['is_media'] and not m['is_deleted']]

# Helper: get messages per person
def msgs_of(sender, msg_list=real_msgs):
    return [m for m in msg_list if m['sender'] == sender]

# ARCHETYPE 1: THE SPAMMER
# Avg consecutive burst > 3 (back-to-back without others speaking)

def score_spammer(sender):
    bursts = []
    current_burst = 0
    for m in messages:
        if m['sender'] == sender:
            current_burst += 1
        else:
            if current_burst > 0:
                bursts.append(current_burst)
            current_burst = 0
    if current_burst > 0:
        bursts.append(current_burst)
    return sum(bursts) / len(bursts) if bursts else 0


# ARCHETYPE 2: THE GROUP MOM
# Highest count of caring keywords

CARING_KEYWORDS = [
    'okay', 'safe', 'eat', 'sleep', 'take care', 'are you',
    'please', 'reminder', 'drink water', "don't forget",
    'careful', 'rest', 'okay', 'alright', 'fine', 'well',
    'health', 'water', 'food', 'stay'
]

def score_group_mom(sender):
    count = 0
    for m in msgs_of(sender):
        text_lower = m['text'].lower()
        for kw in CARING_KEYWORDS:
            count += text_lower.count(kw)
    return count


# ARCHETYPE 3: THE NIGHT OWL
# % of messages between 23:00 and 04:59

def score_night_owl(sender):
    person_msgs = msgs_of(sender)
    if not person_msgs:
        return 0
    night = sum(1 for m in person_msgs if m['datetime'].hour >= 23 or m['datetime'].hour <= 4)
    return (night / len(person_msgs)) * 100

# ARCHETYPE 4: THE STORYTELLER
# Average words per message > 30

def score_storyteller(sender):
    person_msgs = msgs_of(sender)
    if not person_msgs:
        return 0
    total_words = sum(len(m['text'].split()) for m in person_msgs)
    return total_words / len(person_msgs)


# ARCHETYPE 5: THE DRAMA QUEEN
# % of messages that are ALL CAPS or have 2+ exclamation marks

def score_drama_queen(sender):
    person_msgs = msgs_of(sender)
    if not person_msgs:
        return 0
    drama_count = 0
    for m in person_msgs:
        text = m['text']
        # Skip very short messages under 3 chars
        alpha_only = ''.join(c for c in text if c.isalpha())
        if len(alpha_only) < 3:
            continue
        if alpha_only.isupper() or text.count('!') >= 2:
            drama_count += 1
    return (drama_count / len(person_msgs)) * 100


# ARCHETYPE 6: THE GHOST
# Silent on more than 60% of days

def score_ghost(sender):
    silent = total_days - len(active_days[sender])
    return (silent / total_days) * 100


# ARCHETYPE 7: THE COMEDIAN
# % of messages containing lol/lmao/haha/rofl/lmfao

COMEDY_WORDS = ['lol', 'lmao', 'haha', 'rofl', 'lmfao', 'hehe', 'xD']

def score_comedian(sender):
    person_msgs = msgs_of(sender)
    if not person_msgs:
        return 0
    count = sum(1 for m in person_msgs
                if any(w in m['text'].lower() for w in COMEDY_WORDS))
    return (count / len(person_msgs)) * 100


# ARCHETYPE 8: THE QUESTION MASTER
# More than 25% of messages end with '?'

def score_question_master(sender):
    person_msgs = msgs_of(sender)
    if not person_msgs:
        return 0
    count = sum(1 for m in person_msgs if m['text'].strip().endswith('?'))
    return (count / len(person_msgs)) * 100


# BONUS ARCHETYPE 9: THE LATE NIGHT PHILOSOPHER
# Uses deep/existential words after midnight

DEEP_WORDS = ['life', 'meaning', 'time', 'exist', 'wonder', 'reality',
              'dream', 'future', 'purpose', 'feel', 'soul', 'truth',
              'universe', 'question', 'why', 'dread', 'thought']

def score_philosopher(sender):
    count = 0
    for m in msgs_of(sender):
        if m['datetime'].hour >= 0 and m['datetime'].hour <= 4:
            text_lower = m['text'].lower()
            for word in DEEP_WORDS:
                if word in text_lower:
                    count += 1
    return count

# ASSIGN ARCHETYPES


archetypes = {
    'THE SPAMMER':            score_spammer,
    'THE GROUP MOM':          score_group_mom,
    'THE NIGHT OWL':          score_night_owl,
    'THE STORYTELLER':        score_storyteller,
    'THE DRAMA QUEEN':        score_drama_queen,
    'THE GHOST':              score_ghost,
    'THE COMEDIAN':           score_comedian,
    'THE QUESTION MASTER':    score_question_master,
    'THE LATE NIGHT PHILOSOPHER': score_philosopher,
}

# Compute scores for every person x archetype
all_scores = {}
for sender in participants:
    all_scores[sender] = {}
    for name, fn in archetypes.items():
        all_scores[sender][name] = fn(sender)

# Assign with priority order — most "obvious" archetypes first
PRIORITY_ORDER = [
    'THE GHOST',
    'THE NIGHT OWL',
    'THE DRAMA QUEEN',
    'THE STORYTELLER',
    'THE GROUP MOM',
    'THE SPAMMER',
    'THE COMEDIAN',
    'THE QUESTION MASTER',
    'THE LATE NIGHT PHILOSOPHER',
]

assigned = {}
claimed  = {}

for arch in PRIORITY_ORDER:
    best_sender = None
    best_score = -1
    for sender in participants:
        if sender not in assigned:
            s = all_scores[sender][arch]
            if s > best_score:
                best_score = s
                best_sender = sender
    if best_sender:
        assigned[best_sender] = (arch, best_score)
        claimed[arch] = best_sender


# Print archetype details
def archetype_detail(sender, arch, score):
    """Returns a readable detail string for each archetype."""
    if arch == 'THE SPAMMER':
        return f'avg {score:.1f} msgs in a row'
    elif arch == 'THE GROUP MOM':
        return f'caring keyword score: {int(score)}'
    elif arch == 'THE NIGHT OWL':
        return f'{score:.1f}% msgs between 23h-04h'
    elif arch == 'THE STORYTELLER':
        return f'avg {score:.1f} words per msg'
    elif arch == 'THE DRAMA QUEEN':
        return f'{score:.1f}% ALL-CAPS / multi-exclamation msgs'
    elif arch == 'THE GHOST':
        return f'silent on {total_days - len(active_days[sender])} of {total_days} days'
    elif arch == 'THE COMEDIAN':
        return f'{score:.1f}% msgs with laugh words'
    elif arch == 'THE QUESTION MASTER':
        return f'{score:.1f}% msgs end with ?'
    elif arch == 'THE LATE NIGHT PHILOSOPHER':
        return f'deep word score after midnight: {int(score)}'
    return ''

print('  PERSONALITY ARCHETYPES')
print(f'  {"=" * 55}')
for sender in sorted_senders:
    arch, score = assigned[sender]
    detail = archetype_detail(sender, arch, score)
    print(f'  {sender:<8}  →  {arch:<28}  ({detail})')

print()
print('  DEBUG: ALL SCORES')
print(f'  {"-" * 70}')
header = f'  {"":10}' + ''.join(f'{a[:8]:>10}' for a in archetypes.keys())
print(header)
for sender in sorted_senders:
    row = f'  {sender:<10}' + ''.join(f'{all_scores[sender][a]:>10.1f}' for a in archetypes.keys())
    print(row)

  PERSONALITY ARCHETYPES
  Rahul     →  THE SPAMMER                   (avg 4.5 msgs in a row)
  Priya     →  THE GROUP MOM                 (caring keyword score: 835)
  Neha      →  THE DRAMA QUEEN               (63.3% ALL-CAPS / multi-exclamation msgs)
  Aman      →  THE NIGHT OWL                 (80.4% msgs between 23h-04h)
  Karan     →  THE STORYTELLER               (avg 57.0 words per msg)
  Vikas     →  THE GHOST                     (silent on 44 of 60 days)

  DEBUG: ALL SCORES
  ----------------------------------------------------------------------
              THE SPAM  THE GROU  THE NIGH  THE STOR  THE DRAM  THE GHOS  THE COME  THE QUES  THE LATE
  Rahul            4.5       0.0      13.5       2.6       0.0       0.0       3.2       3.8       0.0
  Priya            1.7     835.0       1.3       5.0       0.0       0.0       0.0      29.5       0.0
  Neha             2.6      53.0       4.8       5.3      63.3       0.0       0.0       6.2       0.0
  Aman             2.7   

In [ ]:
# Feature 8: The Final GroupDNA Report
# One clean print from top to bottom — this is the screenshot

def print_separator(char='=', width=62):
    print('  ' + char * width)

def print_section(title):
    print()
    print_separator('-')
    print(f'  {title}')
    print_separator('-')

#   HEADER
print_separator('=')
print(f'  {"GROUPDNA REPORT":^60}')
print(f'  {"Hostel Bois 4ever":^60}')
print(f'  {f"{total_days} days  •  {total_msgs:,} messages  •  {len(participants)} members":^60}')
print_separator('=')

#   GROUP OVERVIEW

print_section(' GROUP OVERVIEW')
print(f'  {"Period":<22}: {first_date.strftime("%d %B %Y")} to {last_date.strftime("%d %B %Y")}')
print(f'  {"Total messages":<22}: {total_msgs:,}')
print(f'  {"Participants":<22}: {len(participants)}')
print(f'  {"System messages skipped":<22}: {system_count}')
print(f'  {"Media messages":<22}: {media_count}')
print(f'  {"Deleted messages":<22}: {deleted_count}')


#   MESSAGES PER PERSON

print_section('  MESSAGES PER PERSON')
for sender in sorted_senders:
    count = msg_count[sender]
    pct = (count / total_msgs) * 100
    bar = make_bar(count, max_count, bar_width=20)
    print(f'  {sender:<8}  {bar}  {count:>4}  ({pct:>4.1f}%)')


#   BUSIEST DAY & HOUR

print_section(' ACTIVITY PEAKS')
print(f'  {"Busiest day":<22}: {busiest_day_str} ({busiest_day_count} messages)')
print(f'  {"Busiest hour":<22}: {busiest_hour:02d}:00 - {busiest_hour+1:02d}:00  (avg {avg_per_day:.0f} msgs/day)')


#   HEATMAP

print_section(' ACTIVITY HEATMAP  (columns = hours 00 to 23)')
render_heatmap(heatmap, person_order)


#   TOP WORDS

print_section("  THIS GROUP'S FAVOURITE WORDS")
for word in top_words:
    count = word_freq[word]
    bar = make_bar(count, max_word_count, bar_width=22)
    print(f'  {word:<12}  {bar}  {count}')

print()
print(f'  TOP 5 WORDS PER PERSON')
print(f'  {"·" * 55}')
for sender in sorted_senders:
    words_str = ',  '.join(top_per_person[sender])
    print(f'  {sender:<8}  →  {words_str}')


#   RESPONSE PATTERNS

print_section('  RESPONSE PATTERNS')
print(f'  Fastest replier  :  {sorted_by_response[0]:<8} (avg {format_time(avg_response[sorted_by_response[0]])})')
print(f'  Slowest replier  :  {sorted_by_response[-1]:<8} (avg {format_time(avg_response[sorted_by_response[-1]])})')
print()
print(f'  {"PERSON":<10}  {"AVG RESPONSE":>15}  {"ACTIVE DAYS":>12}')
print(f'  {"·" * 45}')
for sender in sorted_by_response:
    active = len(active_days[sender])
    print(f'  {sender:<10}  {format_time(avg_response[sender]):>15}  {active:>8} / {total_days}')


#   SILENT STREAKS

print_section('  LONGEST SILENT STREAKS')
for sender in sorted_by_streak:
    streak = silent_streaks[sender]
    if streak == 0:
        print(f'  {sender:<10}  :  0 days  (never went silent)')
    else:
        s, e = streak_details[sender]
        s_fmt = datetime.strptime(s, '%Y-%m-%d').strftime('%d %b')
        e_fmt = datetime.strptime(e, '%Y-%m-%d').strftime('%d %b')
        print(f'  {sender:<10}  :  {streak} days  ({s_fmt} to {e_fmt})')


#   ARCHETYPES

print_section(' PERSONALITY ARCHETYPES')
for sender in sorted_senders:
    arch, score = assigned[sender]
    detail = archetype_detail(sender, arch, score)
    print(f'  {sender:<8}  →  {arch:<30}  ({detail})')


#   FOOTER

print()
print_separator('=')
print(f'  {"Generated by GroupDNA  •  Built with Python + NumPy":^60}')
print(f'  {"No pandas. No matplotlib. Just fundamentals.":^60}')
print_separator('=')

                        GROUPDNA REPORT                       
                       Hostel Bois 4ever                      
            60 days  •  3,174 messages  •  6 members          

  --------------------------------------------------------------
   GROUP OVERVIEW
  --------------------------------------------------------------
  Period                : 01 April 2024 to 30 May 2024
  Total messages        : 3,174
  Participants          : 6
  System messages skipped: 4
  Media messages        : 32
  Deleted messages      : 15

  --------------------------------------------------------------
    MESSAGES PER PERSON
  --------------------------------------------------------------
  Rahul     ████████████████████   953  (30.0%)
  Priya     ███████████████░░░░░   718  (22.6%)
  Neha      █████████████░░░░░░░   635  (20.0%)
  Aman      ██████████░░░░░░░░░░   490  (15.4%)
  Karan     ███████░░░░░░░░░░░░░   354  (11.2%)
  Vikas     ░░░░░░░░░░░░░░░░░░░░    24  ( 0.8%)

  --------------